# Notebook 01 - Prepare Kirundi SFT Dataset

This notebook loads the configured `ptrdvn/kakugo-run` split from Hugging Face, extracts user/assistant pairs, removes visible `<think>...</think>` reasoning traces from assistant responses, and writes local dataset files for the next notebooks.

**Files read:**
- [`../configs/project.yaml`](../configs/project.yaml) - dataset ID, split, optional sample size, random seed, and output paths.
- [`ptrdvn/kakugo-run`](https://huggingface.co/datasets/ptrdvn/kakugo-run) - source Hugging Face dataset loaded by this notebook.

**Files written:**
- [`../data/raw/kakugo_run.jsonl`](../data/raw/kakugo_run.jsonl) - cleaned local dataset with metadata.
- [`../data/processed/kakugo_adaption_input.csv`](../data/processed/kakugo_adaption_input.csv) - prompt/response CSV for Adaption.
- [`../data/processed/kakugo_raw_sft.jsonl`](../data/processed/kakugo_raw_sft.jsonl) - chat-format JSONL for the raw SFT run.

In [ ]:
import json
import sys
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs' / 'project.yaml').exists():
            return candidate
    raise FileNotFoundError(
        'Run this notebook from inside the adaption-kirundi-sft-starter repo.'
    )


ROOT = find_repo_root(Path.cwd())
SRC = str(ROOT / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from kirundi_sft_starter.data import prepare_kakugo_subset
from kirundi_sft_starter.utils import load_env, load_yaml

load_env()

In [ ]:
project_config = load_yaml(ROOT / 'configs/project.yaml')

sft_config = project_config['datasets']['sft']

In [ ]:
print(json.dumps(sft_config, indent=2))

## Load and normalize the configured dataset

`sample_size: null` in `configs/project.yaml` uses the full Hugging Face split. Set `sample_size` to a positive integer, such as `200`, when you want a quick proof-of-concept run.

In [ ]:
sft_df = prepare_kakugo_subset(project_config)

sft_df.head()

In [ ]:
print(f'Prepared examples: {len(sft_df)}')

print('Adaption CSV:', ROOT / sft_config['adaption_input_path'])

print('Raw SFT JSONL:', ROOT / sft_config['raw_sft_path'])

## Inspect saved files

Reload the two downstream files from disk and compare their shape before moving on.

In [ ]:
adaption_csv_path = ROOT / sft_config['adaption_input_path']
raw_sft_jsonl_path = ROOT / sft_config['raw_sft_path']

adaption_saved_df = pd.read_csv(adaption_csv_path)
with raw_sft_jsonl_path.open('r', encoding='utf-8') as f:
    raw_sft_saved_rows = [json.loads(line) for line in f if line.strip()]

saved_files_summary = pd.DataFrame(
    [
        {
            'file': 'kakugo_adaption_input.csv',
            'format': 'CSV',
            'rows': len(adaption_saved_df),
            'columns_or_fields': ', '.join(adaption_saved_df.columns),
            'path': str(adaption_csv_path.relative_to(ROOT)),
        },
        {
            'file': 'kakugo_raw_sft.jsonl',
            'format': 'JSONL',
            'rows': len(raw_sft_saved_rows),
            'columns_or_fields': 'messages, metadata',
            'path': str(raw_sft_jsonl_path.relative_to(ROOT)),
        },
    ]
)

saved_files_summary

In [ ]:
adaption_saved_df.head(3)

In [ ]:
raw_sft_preview_df = pd.DataFrame(
    [
        {
            'example_id': row.get('metadata', {}).get('example_id'),
            'user': row['messages'][0]['content'],
            'assistant': row['messages'][1]['content'],
        }
        for row in raw_sft_saved_rows[:3]
    ]
)

raw_sft_preview_df